In [1]:
# restart kernel
from IPython.core.display import HTML
HTML("<script>Jupyter.notebook.kernel.restart()</script>")

import math, random, pickle, os, copy, itertools, sys, gpytorch
import torch, logging, scipy.io
import numpy as np
import pandas as pd
import seaborn as sns
from numpy import random
from datetime import datetime

import matplotlib.pyplot as plt
from collections.abc import Iterable
%matplotlib inline

BASE_DIR = os.path.dirname(os.path.dirname(os.getcwd()))
sys.path.insert(1, BASE_DIR)

import config
config.init()
from config import device
from server.MAML_server import MAMLRegression
from utils.analyze_results import *
from utils.assistive_functions import load_trained_models, softplus_inverse
from utils.pf_scheduler import PriorFactorScheduler
from utils.train_models import train_models

random_seed = 5
random.seed(random_seed)
np.random.seed(random_seed)
random_state = np.random.RandomState(random_seed)

if torch.cuda.is_available() and not str(device)=='cpu':
    print('[INFO] running on GPU core ' + str(torch.cuda.current_device()))
else:
    print('[WARN] running on CPU')


[INFO] running on GPU core 2


In [2]:
exp_name = 'PV_UniModal'         # PV_BiModal, PV_UniModal_augmented', PV_UniModal
save_maml = True
methods = [
    'lin', 
    'nn4x2', 'nn16x2', 'nn32x2',
    'nn4x4', 'nn16x4', 'nn32x4',
] 
methods_to_run = ['nn4x4'] # best in Bi and uni: 4x4


# Load data
* vary the tilt of the installation and the azimuthal orientation
* same coordinates and altitude of central Lausanne

In [3]:
# ------ LOAD DATA ------
# NOTE: env generated in notebook 1_visualization
filename_env = config.PVDATA_DIR + '/'+ exp_name+"_env"
file = open(filename_env, 'rb')
env_dict = pickle.load(file)
msg = '[INFO] loaded data for {:2.0f} clients'.format(env_dict['num_clients'])
print(msg)
file.close()
num_clients = env_dict['num_clients'] 
print('\n'+env_dict['info'])

# ----- SELECT A SUBSET OF CLIENTS -----
clients_subset= [12, 14, 15, 17, 22]
print('Subset of clients for demonstration: ', clients_subset)

num_features = len(env_dict['feature_names'])
print('\n[INFO] {:2.0f} features: '.format(
    len(env_dict['feature_names'])), *env_dict['feature_names'])
    

[INFO] loaded data for 24 clients

24 households at Lausanne - tilt_std: 5.0, az_std: 15.0, weather_dev: 0.1, irrad_std: 0.2, altitude_dev: 0.1, shadow_peak_red: 0.8, different module_name, different inverter_name, 
Subset of clients for demonstration:  [12, 14, 15, 17, 22]

[INFO] 15 features:  H_sun T2m WS10m station_irrad_direct_prev lag 1 lag 2 lag 4 lag 18 lag 20 lag 22 lag 43 lag 70 lag 74 lag 121 lag 145


In [4]:
if len(methods_to_run)>0:
    models_MAML = {key: {'sml':None, '1y':None} for key in methods_to_run}
    results_MAML = {key: {'sml':None, '1y':None} for key in methods_to_run}
    for scenario_name_fl in ['sml', '1y']:
        clients_data = env_dict['train_scenarios'][scenario_name_fl]['clients_data']
        meta_train_data = []
        meta_test_data = []
        for client_num in np.arange(num_clients):
            meta_train_data.append((clients_data[client_num][0], clients_data[client_num][1]))
        for method in methods_to_run:
            if method=='lin':
                continue
            else:
                print('\n' + method + ' in ' + scenario_name_fl)
                num_layers = int(method.split('x')[-1])
                num_neurons = int(method.split('x')[0][2:])
                layer_sizes = (num_neurons,)*num_layers
                meta_learner = MAMLRegression(
                    meta_train_data, task_batch_size=5, num_iter_fit=5000, layer_sizes=layer_sizes)
                meta_learner.meta_fit(clients_data, log_period=500)
                results_MAML[method][scenario_name_fl] = meta_learner.eval_datasets(clients_data, get_full_list=True)
                models_MAML[method][scenario_name_fl] = meta_learner.best_params




nn4x4 in sml
[2023-05-16 03:13:02,928 -INFO]  Iter 1/5000 - Loss: 0.588021 - Time 0.79 sec Valid-RSMSE: 1.666 
[2023-05-16 03:13:12,004 -INFO]  Iter 500/5000 - Loss: 0.372493 - Time 9.08 sec Valid-RSMSE: 0.641 
[2023-05-16 03:13:21,096 -INFO]  Iter 1000/5000 - Loss: 0.220604 - Time 9.00 sec Valid-RSMSE: 0.596 
[2023-05-16 03:13:29,984 -INFO]  Iter 1500/5000 - Loss: 0.194834 - Time 8.80 sec Valid-RSMSE: 0.575 
[2023-05-16 03:13:38,903 -INFO]  Iter 2000/5000 - Loss: 0.189956 - Time 8.80 sec Valid-RSMSE: 0.568 
[2023-05-16 03:13:47,865 -INFO]  Iter 2500/5000 - Loss: 0.204510 - Time 8.89 sec Valid-RSMSE: 0.564 
[2023-05-16 03:13:56,876 -INFO]  Iter 3000/5000 - Loss: 0.189453 - Time 8.90 sec Valid-RSMSE: 0.559 
[2023-05-16 03:14:05,858 -INFO]  Iter 3500/5000 - Loss: 0.195017 - Time 8.88 sec Valid-RSMSE: 0.553 
[2023-05-16 03:14:14,822 -INFO]  Iter 4000/5000 - Loss: 0.186761 - Time 8.88 sec Valid-RSMSE: 0.550 
[2023-05-16 03:14:23,837 -INFO]  Iter 4500/5000 - Loss: 0.186907 - Time 8.91 sec 

# New Clients

In [5]:
exp_name_ptc = exp_name
exp_name_new = exp_name_ptc + '_NewClients'

# ----- SET UP LOGGER -----
filename_env_new = config.PVDATA_DIR + '/'+ exp_name_ptc +"_new_clients_env"
filename_res_new = os.path.join(os.getcwd(), "saved_results", exp_name_new)


# ------ LOAD META_TEST DATA ------
file = open(filename_env_new, 'rb')
env_dict_new = pickle.load(file)
msg = '[INFO] loaded data for {:2.0f} clients'.format(env_dict_new['num_clients'])
print(msg)
file.close()
num_clients_new = env_dict_new['num_clients'] 
print('\n'+env_dict_new['info'])

clients_data_new = env_dict_new['train_scenarios']['sml']['clients_data']
meta_train_data_new = []
meta_test_data_new = []
for client_num in np.arange(num_clients_new):
    meta_train_data_new.append((clients_data_new[client_num][0], clients_data_new[client_num][1]))
    meta_test_data_new.append((clients_data_new[client_num][2], clients_data_new[client_num][3]))



[INFO] loaded data for 24 clients

24 households at Lausanne - tilt_std: 5.0, az_std: 15.0, weather_dev: 0.1, irrad_std: 0.2, altitude_dev: 0.1, shadow_peak_red: 0.8, different module_name, different inverter_name, 


In [6]:
if len(methods_to_run)>0:
    results_MAML_new = copy.deepcopy(results_MAML)
    for scenario_name_fl in ['sml', '1y']:
        for method in methods_to_run:
            if method=='lin':
                continue
            if models_MAML[method][scenario_name_fl] is None: 
                continue
            print('\n'+method+' in '+scenario_name_fl)
            # reconstruct learner
            num_layers = int(method.split('x')[-1])
            num_neurons = int(method.split('x')[0][2:])
            layer_sizes = (num_neurons,)*num_layers
            meta_learner = MAMLRegression(
                meta_train_data_new, task_batch_size=5, num_iter_fit=5000, layer_sizes=layer_sizes)
            meta_learner.initial_params = [torch.nn.Parameter(torch.tensor(param).to(device)) for param in models_MAML[method][scenario_name_fl]]
            # eval
            results_MAML_new[method][scenario_name_fl] = meta_learner.eval_datasets(clients_data_new, get_full_list=True)
            rsmse_test_new = results_MAML_new[method][scenario_name_fl]['rsmse']
            print(
                np.mean(results_MAML[method][scenario_name_fl]['rsmse']),
                np.mean(rsmse_test_new), 
                # np.quantile(rsmse_test_new, .25), 
                # np.quantile(rsmse_test_new, .50), np.quantile(rsmse_test_new, .75)
            )


nn4x4 in sml
0.545826699047857 0.5271684811844929

nn4x4 in 1y
0.6017741885075457 0.5591315079815287


In [7]:
# To remove
print('sml existing', np.mean(results_MAML['nn4x4']['sml']['rsmse']))
# BiModal: [0.5198103939697368, 0.5080354630029575, 0.5133364673716602, 0.5259531514885433, 0.548085263707498]
# Uni:     [0.568286583147538, 0.5552673549013596, 0.5771969233939762, 0.5853242865258677, 0.554966728383184]
print('1y  existing', np.mean(results_MAML['nn4x4']['1y']['rsmse']))
# BiModal: [0.5232343341294651, 0.5271332453300626, 0.5327929826334136, 0.5250287029008597, 0.5096271770447632]
# Uni:     [0.5859636866974344, 0.5735252663992452, 0.5670678235617824, 0.586841881025424, 0.5873462883031865]
print('sml new     ', np.mean(results_MAML_new['nn4x4']['sml']['rsmse']))
# BiModal:  [0.5186647386089885, 0.513772642212023, 0.5224044290331805, 0.520295779226223, 0.5334619407686961]
# UniModal: [0.5472391768618535, 0.532189327529406, 0.5583274480226214, 0.5644486116203647, 0.5264580733737637]
print('1y  new     ', np.mean(results_MAML_new['nn4x4']['1y']['rsmse']))
# BiModal:  [0.5214323239601835, 0.5373312998641454, 0.5443388072544023, 0.5342467432034309, 0.514159453762895]
# UniModal: [0.5581600767960219, 0.5506115682181142, 0.5433789121618879, 0.5466004710359001, 0.5621243948643618]


sml existing 0.545826699047857
1y  existing 0.6017741885075457
sml new      0.5271684811844929
1y  new      0.5591315079815287


In [8]:
a = [0.5581600767960219, 0.5506115682181142, 0.5433789121618879, 0.5466004710359001, 0.5621243948643618]
print('{:1.3f}, {:1.3f}'.format(np.mean(a), 1.96 * np.std(a)))


0.552, 0.014


In [9]:
filename_save = os.path.join(BASE_DIR, 'experiments', 'PV', 'saved_results', exp_name, 'MAML', 'MAML')
if save_maml:
    file = open(filename_save, 'wb')
    pickle.dump({'results': results_MAML, 'models':models_MAML, 'results_new':results_MAML_new}, file)
    file.close()
    print('models saved.')
else:
    file = open(filename_save, 'rb')
    res = pickle.load(file)
    file.close()
    results_MAML_new = res['results_new']
    results_MAML = res['results']
    models_MAML = res['models']

models saved.


5:
nn4x2 in sml
0.5198103939697368 0.5186647386089885

nn16x2 in sml
0.578062151351349 0.5226248316108938

nn32x2 in sml
0.5730231695881834 0.5169378741713349

nn4x4 in sml
0.5260404315455733 0.5227809982504991

nn16x4 in sml
0.5703883281709586 0.5228498766694227

nn32x4 in sml
0.5623443201653552 0.526428899654507

nn4x2 in 1y
0.5232343341294651 0.5214323239601835

nn16x2 in 1y
0.5493125335919756 0.5192277579392194

nn32x2 in 1y
0.5590345594537472 0.5237550094993045

nn4x4 in 1y
0.5411756345121507 0.5395890110984914

nn16x4 in 1y
0.5523013335961507 0.5195808766908913

nn32x4 in 1y
0.5723329558598986 0.516144560780893